<a href="https://colab.research.google.com/github/cafauzi13/ESG_SentimentAnalysis/blob/main/scripts/Augmentasi_LLM_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Augmentasi Data — ESG Sentiment Analysis
**Strategi anti-limit Groq API (Llama 3):**
- Hard cap: 12 request/menit (aman di bawah limit 15 RPM)
- Exponential backoff + jitter saat kena 429
- Checkpoint otomatis ke Google Drive setiap 5 artikel
- Resume otomatis jika session mati di tengah jalan
- Teks dipotong 600 karakter untuk hemat token
- Estimasi total waktu ditampilkan sebelum mulai


## 0. Install & Mount Drive

In [1]:
!pip install -q groq scikit-learn

import os

try:
    # 1. Deteksi lingkungan Google Colab
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/ESG/'
    print("✅ Berjalan di lingkungan Google Colab. Drive berhasil di-mount!")
except ImportError:
    # 2. Berjalan di lingkungan Lokal
    DRIVE_DIR = 'data/'
    print("✅ Berjalan di lingkungan Lokal. Menggunakan path folder data/ lokal secara otomatis!")

CHECKPOINT_PATH  = DRIVE_DIR + 'augmented_checkpoint.csv'
TRAIN_OUT_PATH   = DRIVE_DIR + 'train_set_augmented.csv'
TEST_OUT_PATH    = DRIVE_DIR + 'test_set_asli.csv'

os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Path output dikonfigurasi: {DRIVE_DIR}")


✅ Berjalan di lingkungan Lokal. Menggunakan path folder data/ lokal secara otomatis!
Path output dikonfigurasi: data/


## 1. Setup Groq Client


In [2]:
import os
from groq import Groq

try:
    # 1. Deteksi lingkungan Google Colab
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("✅ Berjalan di lingkungan Google Colab Cloud Mode. API Key berhasil dimuat secara aman via Secrets!")
except ImportError:
    # 2. Jika ImportError, berarti berjalan di Lokal (Antigravity IDE)
    # Menggunakan fallback API Key milik Bara secara langsung
    os.environ["GROQ_API_KEY"] = "gsk_REDACTED_FOR_SECURITY_KEY_gsk"
    print("✅ Berjalan di lingkungan Lokal Antigravity IDE Mode. Menggunakan fallback API Key lokal secara otomatis!")

# Inisialisasi client global menggunakan API key yang telah ter-set di environment
client = Groq()

MODEL_NAME = 'llama-3.3-70b-versatile'

# Test koneksi
test_resp = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Balas hanya dengan kata OK"}
    ]
)
print(f'✅ Groq OK: {test_resp.choices[0].message.content.strip()}')


✅ Berjalan di lingkungan Lokal Antigravity IDE Mode. Menggunakan fallback API Key lokal secara otomatis!


✅ Groq OK: OK


## 2. Load Data & Train/Test Split

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

local_paths = [
    'IndoBERT.csv',
    'data/IndoBERT.csv',
    '../data/IndoBERT.csv',
    'scripts/IndoBERT.csv'
]

df = None
for path in local_paths:
    if os.path.exists(path):
        try:
            df = pd.read_csv(path)
            print(f'✅ Dataset berhasil di-load secara lokal dari {path}! Total data: {df.shape[0]} baris.')
            break
        except Exception as e:
            print(f'⚠️ Gagal load lokal dari {path}: {e}')

if df is None:
    GITHUB_URL = 'https://raw.githubusercontent.com/cafauzi13/ESG_SentimentAnalysis/main/data/IndoBERT.csv'
    try:
        df = pd.read_csv(GITHUB_URL)
        print(f'✅ Dataset berhasil di-load via GitHub! Total data: {df.shape[0]} baris.')
    except Exception as e:
        print(f'❌ Gagal load data dari GitHub!. Error: {e}')

df = df.dropna(subset=['Isi Berita Clean', 'Sentiment', 'Tag'])
df['Isi Berita Clean'] = df['Isi Berita Clean'].astype(str)
print(f'Total data: {len(df)}')
print(f'\nDistribusi Sentimen:\n{df["Sentiment"].value_counts()}')

# Train/test split — test set TIDAK diaugmentasi
df_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['Sentiment']
)
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

# Simpan test set sekali saja ke Drive
df_test.to_csv(TEST_OUT_PATH, index=False)

print(f'\nTrain: {len(df_train)} | Test: {len(df_test)}')
print(f'\nDistribusi Train:\n{df_train["Sentiment"].value_counts()}')
print(f'✅ Test set disimpan: {TEST_OUT_PATH}')


✅ Dataset berhasil di-load secara lokal dari IndoBERT.csv! Total data: 468 baris.
Total data: 468

Distribusi Sentimen:
Sentiment
Negatif    195
Positif    172
Netral     101
Name: count, dtype: int64

Train: 374 | Test: 94

Distribusi Train:
Sentiment
Negatif    156
Positif    137
Netral      81
Name: count, dtype: int64
✅ Test set disimpan: data/test_set_asli.csv


## 3. Estimasi Waktu & Konfigurasi Rate Limit

In [4]:
import math

# ============================================================
# KONFIGURASI RATE LIMIT — jangan ubah kecuali pakai paid tier
# ============================================================
RPM_LIMIT      = 12    # request per menit (free tier max = 15, kita pakai 12 untuk aman)
SLEEP_BASE     = 60 / RPM_LIMIT   # 5 detik antar request
MAX_CHARS      = 600   # potong teks input untuk hemat token
MAX_OUT_TOKENS = 600   # output token per request
MAX_RETRIES    = 5     # maksimal retry saat kena 429
CHECKPOINT_N   = 5     # simpan checkpoint setiap N artikel

# Hitung kebutuhan augmentasi
max_count  = df_train['Sentiment'].value_counts().max()
total_need = sum(
    max(0, max_count - cnt)
    for cnt in df_train['Sentiment'].value_counts()
)

est_minutes = math.ceil(total_need * SLEEP_BASE / 60)

print('=' * 50)
print('  ESTIMASI AUGMENTASI')
print('=' * 50)
print(f'  Target per kelas  : {max_count}')
print(f'  Total perlu dibuat: {total_need} artikel')
print(f'  Jeda antar request: {SLEEP_BASE:.1f} detik')
print(f'  Estimasi waktu    : ~{est_minutes} menit')
print('=' * 50)
print('\n⚠️  Jangan tutup tab Colab. Jika session mati,')
print('   re-run dari cell ini — checkpoint otomatis resume.')


  ESTIMASI AUGMENTASI
  Target per kelas  : 156
  Total perlu dibuat: 94 artikel
  Jeda antar request: 5.0 detik
  Estimasi waktu    : ~8 menit

⚠️  Jangan tutup tab Colab. Jika session mati,
   re-run dari cell ini — checkpoint otomatis resume.


## 4. Fungsi Augmentasi dengan Backoff

In [5]:
import time
import random

def augment_artikel(text, sentiment, tag, attempt_num=0):
    """
    Generate 1 parafrase artikel.
    Return: string teks baru, atau None jika gagal setelah MAX_RETRIES.
    """
    text_input = str(text)[:MAX_CHARS]

    prompt = (
        f"Kamu adalah parafrase generator untuk dataset riset berita ESG Indonesia.\n"
        f"Artikel asli (sentimen: {sentiment}, kategori ESG: {tag}):\n"
        f"{text_input}\n\n"
        f"Tugas: Buat 1 parafrase artikel di atas. Ketentuan Wajib:\n"
        f"- Pertahankan sentimen {sentiment} secara konsisten. Jangan melembutkan, mengubah, atau menyensor nada kritik/apresiasi dari artikel asli.\n"
        f"- Pertahankan konteks ESG kategori {tag}.\n"
        f"- Gunakan gaya bahasa penulisan berita jurnalistik formal Indonesia (seperti gaya penulisan kantor berita Antara, Kompas, atau Tempo) yang natural.\n"
        f"- Ubah struktur kalimat dan pilihan kata secara mendalam, bukan sekadar mengganti sinonim kata per kata.\n"
        f"- Mulai respons langsung dengan kalimat pertama hasil parafrase. Dilarang keras menggunakan kalimat pengantar, kalimat penutup, sapaan robotik, tanda kutip pembungkus (\"\"), atau catatan kaki apapun.\n"
        f"- HANYA output teks parafrase saja, tanpa penjelasan, tanpa label apapun."
    )

    for attempt in range(MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                max_tokens=MAX_OUT_TOKENS,
                temperature=0.75,
            )
            hasil = resp.choices[0].message.content.strip()
            if len(hasil) >= 100:
                return hasil
            else:
                print(f'    ⚠️ Output terlalu pendek ({len(hasil)} chars), retry {attempt+1}...')
                time.sleep(SLEEP_BASE)

        except Exception as e:
            err_str = str(e).lower()

            if '429' in err_str or 'quota' in err_str or 'rate' in err_str:
                # Exponential backoff dengan jitter
                wait = (2 ** (attempt + 1)) + random.uniform(1, 5)
                print(f'    🚦 Rate limit (429). Tunggu {wait:.0f} detik... (attempt {attempt+1}/{MAX_RETRIES})')
                time.sleep(wait)

            elif 'resource' in err_str or '503' in err_str or '500' in err_str:
                wait = 15 + random.uniform(0, 10)
                print(f'    🔄 Server error. Tunggu {wait:.0f} detik...')
                time.sleep(wait)

            else:
                print(f'    ❌ Error tidak dikenal: {e}')
                time.sleep(SLEEP_BASE)

    print(f'    ❌ Gagal setelah {MAX_RETRIES} percobaan, skip artikel ini.')
    return None


## 5. Jalankan Augmentasi (dengan Resume Otomatis)

In [6]:
# ============================================================
# RESUME: load checkpoint jika ada
# ============================================================
if os.path.exists(CHECKPOINT_PATH):
    df_checkpoint = pd.read_csv(CHECKPOINT_PATH)
    augmented_rows = df_checkpoint.to_dict('records')
    print(f'🔄 Resume dari checkpoint: {len(augmented_rows)} artikel sudah ada')
    print(f'   Distribusi checkpoint:\n{df_checkpoint["Sentiment"].value_counts().to_string()}')
else:
    augmented_rows = []
    print('🆕 Mulai augmentasi dari awal')

# ============================================================
# AUGMENTASI PER KELAS SENTIMEN
# ============================================================
max_count = df_train['Sentiment'].value_counts().max()

for sentiment in df_train['Sentiment'].unique():
    df_kelas  = df_train[df_train['Sentiment'] == sentiment]
    sudah_ada = sum(1 for r in augmented_rows if r.get('Sentiment') == sentiment)
    kekurangan = max_count - len(df_kelas) - sudah_ada

    if kekurangan <= 0:
        print(f'\n✅ Sentimen "{sentiment}": sudah cukup ({len(df_kelas)} asli + {sudah_ada} sintetis), skip.')
        continue

    print(f'\n{"="*55}')
    print(f'  Sentimen: "{sentiment}" | Perlu generate: {kekurangan} artikel')
    print(f'  (Asli: {len(df_kelas)}, sudah di-checkpoint: {sudah_ada})')
    print(f'{"="*55}')

    # Sampling dengan replace agar bisa generate lebih banyak dari data asli
    samples = df_kelas.sample(n=kekurangan, replace=True, random_state=42).reset_index(drop=True)
    berhasil, gagal = 0, 0

    for i, row in samples.iterrows():
        nomor = i + 1
        print(f'  [{nomor:3d}/{kekurangan}] Generating "{sentiment}"... ', end='', flush=True)

        teks_baru = augment_artikel(
            row['Isi Berita Clean'],
            row['Sentiment'],
            row['Tag']
        )

        if teks_baru:
            new_row = row.to_dict()
            new_row['Isi Berita Clean'] = teks_baru
            new_row['is_augmented'] = True
            augmented_rows.append(new_row)
            berhasil += 1
            print(f'✅ ({len(teks_baru)} chars)')
        else:
            gagal += 1
            print('⚠️ Skip')

        # Checkpoint ke Drive setiap CHECKPOINT_N artikel
        if len(augmented_rows) % CHECKPOINT_N == 0 and augmented_rows:
            pd.DataFrame(augmented_rows).to_csv(CHECKPOINT_PATH, index=False)
            print(f'  💾 Checkpoint disimpan ({len(augmented_rows)} artikel total)')

        # Jeda wajib antar request
        time.sleep(SLEEP_BASE)

    print(f'\n  Hasil "{sentiment}": {berhasil} berhasil, {gagal} gagal')

# Checkpoint final
if augmented_rows:
    pd.DataFrame(augmented_rows).to_csv(CHECKPOINT_PATH, index=False)
    print(f'\n💾 Checkpoint final disimpan: {len(augmented_rows)} artikel sintetis')


🔄 Resume dari checkpoint: 94 artikel sudah ada
   Distribusi checkpoint:
Sentiment
Netral     75
Positif    19

✅ Sentimen "Netral": sudah cukup (81 asli + 75 sintetis), skip.

✅ Sentimen "Positif": sudah cukup (137 asli + 19 sintetis), skip.

✅ Sentimen "Negatif": sudah cukup (156 asli + 0 sintetis), skip.

💾 Checkpoint final disimpan: 94 artikel sintetis


## 6. Gabungkan & Simpan Train Set Final

In [7]:
if not augmented_rows:
    print('❌ Tidak ada artikel sintetis. Periksa API key dan koneksi.')
else:
    df_train['is_augmented'] = False
    df_aug = pd.DataFrame(augmented_rows)
    df_aug['is_augmented'] = True

    df_train_final = pd.concat([df_train, df_aug], ignore_index=True)
    df_train_final = df_train_final.sample(frac=1, random_state=42).reset_index(drop=True)

    df_train_final.to_csv(TRAIN_OUT_PATH, index=False)

    print('=' * 55)
    print('  TRAIN SET FINAL')
    print('=' * 55)
    print(f'  Total artikel    : {len(df_train_final)}')
    print(f'  Asli             : {df_train_final["is_augmented"].eq(False).sum()}')
    print(f'  Sintetis         : {df_train_final["is_augmented"].eq(True).sum()}')
    print(f'\n  Distribusi Sentimen:')
    print(df_train_final['Sentiment'].value_counts().to_string())
    print(f'\n  Distribusi Tag:')
    print(df_train_final['Tag'].value_counts().to_string())
    print('=' * 55)
    print(f'\n✅ Disimpan: {TRAIN_OUT_PATH}')
    print(f'✅ Test set : {TEST_OUT_PATH}')
    print('\n→ Lanjut ke klasifikasi.ipynb')


  TRAIN SET FINAL
  Total artikel    : 468
  Asli             : 374
  Sintetis         : 94

  Distribusi Sentimen:
Sentiment
Netral     156
Negatif    156
Positif    156

  Distribusi Tag:
Tag
Social           120
Environment       91
Investigation     87
Governance        85
Finance           85

✅ Disimpan: data/train_set_augmented.csv
✅ Test set : data/test_set_asli.csv

→ Lanjut ke klasifikasi.ipynb


## 7. Verifikasi Kualitas Augmentasi (Opsional)
Cek beberapa sampel hasil augmentasi secara manual

In [8]:
import textwrap

df_aug_check = df_train_final[df_train_final['is_augmented'] == True].sample(3, random_state=1)

for i, (_, row) in enumerate(df_aug_check.iterrows()):
    print(f'\n{"─"*55}')
    print(f'  Sampel #{i+1} | Sentimen: {row["Sentiment"]} | Tag: {row["Tag"]}')
    print(f'{"─"*55}')
    print(textwrap.fill(str(row['Isi Berita Clean'])[:400], width=70))
    print('  ...')



───────────────────────────────────────────────────────
  Sampel #1 | Sentimen: Netral | Tag: Environment
───────────────────────────────────────────────────────
Dalam upaya memperkenalkan dan mempromosikan komoditas kelapa sawit
kepada generasi muda, GAPKI bekerja sama dengan Astra Agro dan
Forwatan mengadakan kegiatan edukatif di Panti Asuhan Yatim Piatu
Yayasan Al-Mukhilisin Jakarta. Ketua Kompartemen Media Relation GAPKI,
Fenny Sofyan, berkesempatan berbagi pengetahuan tentang kelapa sawit
dengan cara yang menyenangkan dan mudah dipahami oleh puluhan a
  ...

───────────────────────────────────────────────────────
  Sampel #2 | Sentimen: Netral | Tag: Social
───────────────────────────────────────────────────────
Insiden ledakan hebat terjadi di sebuah fasilitas pengolahan dan
pemurnian stainless steel yang dioperasikan PT Indonesia Tsingshan
Stainless Steel di kawasan industri PT Indonesia Morowali Industrial
Park, Sulawesi Tengah, pada Minggu, 24 Desember 2023, sekitar pukul
06.